In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from os import listdir
from os.path import isfile

In [ ]:
unique_celltypes = pd.read_table("../results/processedData/normalized_mean/unique_celltypes.table", sep=";")
unique_celltypes = unique_celltypes[unique_celltypes["ID"].isna() == False]
unique_celltypes["ID"] = unique_celltypes["ID"].apply(lambda x: x.split("-")[1])

In [ ]:
filepath = "../results/deseq2_rsem/tables/"

experiments = listdir(filepath)
experiment_names = {}
for exp in experiments:
    if "HydraRecolonization_Conventionalized" in exp or "HydraRecolonization_Cvbct" in exp:
        experiment_name = exp.split("_results")[0]
    else:
        experiment_name = exp.split("_vs_")[0]
    experiment_names[exp] = experiment_name

In [ ]:
# spearman correlation procedure
correlation_dict = {}
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        print("[*] Building correlation matrix for celltype: {}".format(celltype))
        correlation_dict[celltype] = {}
        for exp in experiments:
            correlation_dict[celltype][experiment_names[exp]] = []
            log2FoldChange_df = pd.read_csv(filepath + exp)
            log2FoldChange_df["Unnamed: 0"] = log2FoldChange_df["Unnamed: 0"].apply(lambda x: x.split(".")[1])
            log2FoldChange_df.rename(columns={"Unnamed: 0":"ID"}, inplace=True)
            first_df = unique_celltypes[unique_celltypes[celltype] == 1.0].merge(log2FoldChange_df, on="ID")[["ID","log2FoldChange","padj"]]

            for exp_comp in experiments:
                log2FoldChange_df_comparable = pd.read_csv(filepath + exp_comp)
                log2FoldChange_df_comparable["Unnamed: 0"] = log2FoldChange_df_comparable["Unnamed: 0"].apply(lambda x: x.split(".")[1])
                log2FoldChange_df_comparable.rename(columns={"Unnamed: 0":"ID"}, inplace=True)

                comparable_df = unique_celltypes[unique_celltypes[celltype] == 1.0].merge(log2FoldChange_df_comparable, on="ID")[["ID","log2FoldChange","padj"]]
                result_cor, result_pval = stats.spearmanr(first_df.log2FoldChange, comparable_df.log2FoldChange)
                correlation_dict[celltype][experiment_names[exp]].append([experiment_names[exp_comp],result_cor, result_pval])
        print("DONE")

In [ ]:
# plotting procedure
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        rows = len(experiments)
        cols = len(experiments)
        matstat = np.zeros((rows, cols))
        matp = np.zeros((rows, cols))
        experiments_col = []
        for col_idx, exp in enumerate(experiments):
            experiments_col.append(experiment_names[exp])
            for row_idx,res in enumerate(correlation_dict[celltype][experiment_names[exp]]):
                if res[2] <= 0.05:
                    matstat[row_idx,col_idx] = res[1]

        fig, ax = plt.subplots(figsize=(12,8))

        mask = np.triu(np.ones_like(matstat), k=1)
        masked_matrix = np.ma.masked_where(mask == 1, matstat)

        plt.imshow(masked_matrix, cmap="RdBu")
        plt.yticks(np.arange(len(experiments_col)), experiments_col)
        plt.xticks(np.arange(len(experiments_col)), experiments_col, rotation=90) 

        for i in range(masked_matrix.shape[0]):
            for j in range(masked_matrix.shape[1]):
                if masked_matrix[i, j] != 0.0:
                    ax.text(j, i, f'{masked_matrix[i, j]:.2f}', ha='center', va='center', color='black', fontsize=12)
                else:
                    ax.text(j, i, '', ha='center', va='center', color='white')

        plt.title("Spearman Correlation Heatmap - {}".format(celltype))
        plt.colorbar()
        plt.tight_layout()
        plt.savefig("../results/figures/correlation_analysis/correlation_analysis_"+celltype+".jpg", dpi=400)
        plt.close()